# AutoParts Review Triage: from customer reviews and photos to an explained work queue
**BC3415 AI in Accounting and Finance, Assignment: Real-World Applications of Text Classification (Option A: code + video)**

## The business problem
**AutoParts** is an online retailer of automotive parts and accessories (fasteners, wiring, fluids, electronic modules, seat covers, floor mats). It receives thousands of reviews a day, many with customer photos. The company wants to (1) classify reviews as positive / negative / neutral, (2) detect product defects or wrong deliveries from customer photos, and (3) combine both to **automatically flag problematic orders**.

**Why this sector.** In auto parts a bad order is not just a bad review: a part that is defective or **does not fit the vehicle** can leave a car off the road and ends in a return. The typical problems are damaged or defective parts, parts that do not fit or are not compatible, and listings that do not match the part. Defects and wrong parts are often visible in a photo, so text and photos both matter.

## Our answer in one sentence
A star rating or a sentiment score says *that* a customer is unhappy; our **triage engine** says *what went wrong, how urgent it is, why, and what to do*, using the review text and the customer's photo together.

## Pipeline
`review text` -> **cleaning** -> **sentiment model** (calibrated P(negative)) + **issue detector** (damaged, wrong item, not as described, fit / compatibility, delivery, customer service) `+ photo` -> **image evidence** (damage / wrong part) -> **priority score** -> HIGH / MEDIUM / LOW + evidence + suggested action.

## Notebook map
1. Setup  2. Data and preprocessing  3. Sentiment models (comparison)  4. Explainability  5. **Locked test-set evaluation (run once)**  6. Issue detector  7. Image evidence  8. Triage engine and evaluation on unseen random reviews  9. Business value (and 9b: insights for managers)  10. Package for the business app  11. Limitations, ethics, references

## How to run
Runtime > **T4 GPU**, then Run all (about 20-30 minutes). Upload the files listed in the first code cell. Random seeds are fixed (42).

## Evaluation discipline
- Train / validation / **test** sets are separate; **test is scored once, in Section 5**.
- Issue-detector settings are tuned on labelled Sets 1+2 (160 reviews) and evaluated on **Set 3: 100 purely random reviews never used for tuning**. The labelled reviews come from a different part of the data file than the training reviews (no overlap).
- Labels for issue types were annotated by an AI assistant (see Section 11 for the student's verification of a 25-review sample). Small samples are flagged wherever they limit a conclusion.

: this code and analysis were developed with the assistance of Claude (Anthropic) as permitted for this assignment; the student ran and reviewed all outputs and labelled a validation sample.*


In [ ]:
# Files to upload: triage_core.py, triage_engine.py, triage_image.py, triage_insights.py, auto_data.py, labelled_auto.csv, demo_auto_reviews.csv
# (GPU runtime required: this notebook fine-tunes DistilBERT, about 3 minutes)
!pip install -q datasets transformers sentencepiece sentence-transformers scikit-learn huggingface_hub pyarrow gradio joblib
from google.colab import files
up = files.upload()


In [ ]:
import re, io, os, json, random, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from PIL import Image
from datasets import load_dataset
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score, precision_score, recall_score
warnings.filterwarnings("ignore")
import triage_core as tc, triage_engine as te
from triage_core import ISSUES, CODES

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
LABELS = ["negative", "neutral", "positive"]; N_TRAIN = 50_000
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 80)
print("GPU:", torch.cuda.is_available())


## 2. Data and preprocessing
**Data:** public Amazon product reviews from the **Automotive** category (Amazon Reviews 2023, McAuley Lab), streamed from the Hugging Face Hub. The file is read once and split **by position**: the first 300,000 reviews feed the sentiment model (star-balanced: 8,000 per star for training, 1,000 per star each for validation and test); later, separate parts supply the hand-labelled issue sets and the business demo, so nothing overlaps. Star ratings become sentiment: 1-2 stars = negative, 3 = neutral, 4-5 = positive. *Caveat: stars are a proxy for sentiment; real automotive reviews are about 70% five-star, so we balance by star, which means this dataset has far more unhappy reviews than a typical shop.*

**Preprocessing:** drop empties and duplicates; lowercase; strip HTML and URLs; collapse whitespace. **Stop-words are kept** because "not" and "no" flip meaning. **Leakage checks:** any validation/test review that also appears in train is removed.


In [ ]:
import auto_data as ad
print("Streaming automotive reviews from the Hugging Face Hub (about 1-2 minutes)...")
stream_df = ad.stream(0, 480_000)
SPLITS = ad.model_splits(stream_df.iloc[0:300_000])          # star-balanced train / validation / test, no overlap
print("real star mix in the raw data:", stream_df.stars.value_counts(normalize=True).sort_index().round(3).to_dict())
to3 = lambda s: 0 if s <= 1 else (1 if s == 2 else 2)

def prep(split, n=None):
    d = SPLITS[split]
    if n: d = d.iloc[:n]
    df = pd.DataFrame({"text": list(d["text"]), "stars": list(d["label"])})        # stars stored 0..4
    df["y"] = df.stars.map(to3)
    n0 = len(df); df = df.dropna(subset=["text"]); df = df[df.text.str.strip() != ""].drop_duplicates(subset="text").copy()
    df["clean"] = df.text.map(tc.clean); print(f"{split}: {n0} -> {len(df)} after removing empties/duplicates")
    return df.reset_index(drop=True)

train_df, val_df, test_df = prep("train", N_TRAIN), prep("validation"), prep("test")
seen = set(train_df.clean)
n_v, n_t = len(val_df), len(test_df)
val_df = val_df[~val_df.clean.isin(seen)].reset_index(drop=True)
test_df = test_df[~test_df.clean.isin(seen | set(val_df.clean))].reset_index(drop=True)
print(f"leaked rows removed -> val: {n_v - len(val_df)}, test: {n_t - len(test_df)}")
print({"train": len(train_df), "val": len(val_df), "test (LOCKED until section 5)": len(test_df)})
display(pd.DataFrame({n: d.y.map(dict(enumerate(LABELS))).value_counts(normalize=True) for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]}).round(3))
for s in (0, 4):   # sanity check on the label mapping
    print(f"--- stars label {s} ---"); [print("  *", t[:110]) for t in train_df[train_df.stars == s].text.head(2)]

LAB = pd.read_csv("labelled_auto.csv"); LAB = LAB[LAB.problem_order != "?"].reset_index(drop=True)
LAB["issue_codes"] = LAB.issue_codes.fillna(""); LAB["y_problem"] = LAB.problem_order == "Y"
for k in ISSUES: LAB[f"y_{k}"] = LAB.issue_codes.apply(lambda s: CODES[k] in s.split(","))
print("\nhand-checked reviews:", LAB.groupby("set").size().to_dict())
X_train, y_train, X_val, y_val = train_df.clean, train_df.y, val_df.clean, val_df.y
exclude = set(LAB.review.map(tc.clean)) | set(train_df.clean)
cal_df = val_df[~val_df.clean.isin(exclude)].reset_index(drop=True)   # calibration data: never our labelled reviews, never train


## 3. Sentiment models: a fair comparison on the validation set
Models, from simple to advanced: **majority-class baseline**, **TF-IDF + Naive Bayes**, **TF-IDF + Logistic Regression** (class-balanced), **frozen BERT-style sentence embeddings + Logistic Regression** (MiniLM), and a **fine-tuned DistilBERT** (trained here: 1 epoch on 40,000 reviews).
**Metric: macro-F1** (the neutral class is small and hard, so accuracy alone misleads: Naive Bayes has high accuracy but almost never predicts neutral) with **bootstrap 95% confidence intervals**. Selection uses validation data only.


In [ ]:
def boot_ci(y, p, n=1000):
    r = np.random.default_rng(SEED); y, p = np.asarray(y), np.asarray(p)
    s = [f1_score(y[i], p[i], average="macro") for i in (r.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(s, [2.5, 97.5])

val_pred, models = {}, {}
models["Majority class (baseline)"] = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
word_vec = lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000, sublinear_tf=True)
models["TF-IDF + Naive Bayes"] = Pipeline([("f", word_vec()), ("c", MultinomialNB())]).fit(X_train, y_train)
models["TF-IDF + LogReg"] = Pipeline([("f", word_vec()), ("c", LogisticRegression(C=1, max_iter=1000, class_weight="balanced"))]).fit(X_train, y_train)
for n, m in models.items(): val_pred[n] = m.predict(X_val)

from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("all-MiniLM-L6-v2"); N_EMB = 20_000
enc = lambda t: st_model.encode(list(t), batch_size=128, show_progress_bar=False)
emb_lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(enc(X_train.iloc[:N_EMB]), y_train.iloc[:N_EMB])
val_pred["Frozen MiniLM embeddings + LogReg"] = emb_lr.predict(enc(X_val))

sentiment_bert = None
if torch.cuda.is_available():
    sentiment_bert = te.TransformerSentiment(epochs=1).fit(X_train.iloc[:40_000], y_train.iloc[:40_000], cal_df.clean, cal_df.y)   # about 3 min on a T4
    val_pred["Fine-tuned DistilBERT"] = sentiment_bert.predict_label(X_val)
else: print("No GPU: DistilBERT skipped (switch the runtime to a T4 GPU).")

rows = []
for n, p in val_pred.items():
    lo, hi = boot_ci(y_val, p)
    rows.append({"model": n, "accuracy": accuracy_score(y_val, p), "macro-F1": f1_score(y_val, p, average="macro"), "95% CI": f"{lo:.3f}-{hi:.3f}",
                 "F1 neutral": f1_score(y_val, p, average=None)[1]})
comp = pd.DataFrame(rows).set_index("model").sort_values("macro-F1", ascending=False); display(comp.round(3))
print("Best on validation:", comp.index[0])


**Reading the table.** The majority baseline shows what "doing nothing" scores. Naive Bayes wins on accuracy but fails the neutral class. Frozen embeddings lose to TF-IDF: general-purpose sentence vectors are not tuned to sentiment. Fine-tuning a transformer (if present) adapts the whole model to this task; the trade-off is cost and explainability.

**Deployed model (rule fixed in advance).** The triage engine deploys the **fine-tuned DistilBERT** only if it beats TF-IDF + LogReg by at least 0.02 validation macro-F1, otherwise the simpler, faster TF-IDF model. Either way probabilities are **calibrated** (so P(negative) is trustworthy as a risk score) and every decision is explained by deleting words and measuring the drop in confidence. Calibration uses validation reviews that are *not* in our labelled sets.


In [ ]:
sentiment_lr = te.SentimentModel().fit(X_train, y_train, cal_df.clean, cal_df.y)
f_lr = f1_score(y_val, sentiment_lr.predict_label(X_val), average="macro")
f_bert = f1_score(y_val, val_pred["Fine-tuned DistilBERT"], average="macro") if sentiment_bert is not None else None
if f_bert is not None and f_bert >= f_lr + 0.02: sentiment, deployed = sentiment_bert, "Fine-tuned DistilBERT"      # rule fixed in advance
else: sentiment, deployed = sentiment_lr, "TF-IDF + LogReg"
print(f"validation macro-F1: TF-IDF+LogReg {f_lr:.3f} | DistilBERT {'n/a' if f_bert is None else round(f_bert, 3)}  ->  DEPLOYED: {deployed}")

# Calibration check: does "70% negative" really mean ~70% negative?
from sklearn.calibration import calibration_curve
raw = sentiment._raw(cal_df.clean.tolist()); cal = sentiment.predict_proba(cal_df.clean.tolist())
plt.figure(figsize=(4.5, 4.5)); plt.plot([0, 1], [0, 1], "--", color="grey")
for lab_, P in [("raw", raw), ("calibrated", cal)]:
    fp, mp = calibration_curve((cal_df.y.values == 0).astype(int), P[:, 0], n_bins=10); plt.plot(mp, fp, marker="o", label=lab_)
plt.xlabel("predicted P(negative)"); plt.ylabel("actual share negative"); plt.legend(); plt.title("Reliability (negative class)"); plt.show()
print("(calibrators were fitted on this same set, so this plot is illustrative; held-out calibration results are in the project log)")


## 4. Explainability
Two views: **global** (which words push each class) and **per review** (which words drove *this* decision, found by deleting each word and measuring the drop in confidence).


In [ ]:
# global word weights come from the TF-IDF model
vec, clf = sentiment_lr.pipe.named_steps["f"], sentiment_lr.pipe.named_steps["c"]; vocab = np.array(vec.get_feature_names_out())
for i, l in enumerate(LABELS): print(f"{l.upper():9s}:", ", ".join(vocab[np.argsort(clf.coef_[i])[-10:][::-1]]))
for t in ["Arrived late and the box was crushed, very disappointed.", "Love it, fits perfectly and works great!"]:
    p = sentiment.predict_proba([t])[0]; print(f"\n'{t}'\n  P(neg/neu/pos) = {p.round(2)} | driving words: {[(w, round(d, 2)) for w, d in sentiment.explain(t)]}")


## 5. Final evaluation on the locked test set (run once)
Two finalists were declared in advance: **TF-IDF + LogReg** and **fine-tuned DistilBERT**; the one picked by the validation rule above is deployed. Baselines are shown for context. Nothing was tuned on these numbers.


In [ ]:
yt = test_df.y.values; trows = []
def trow(name, p):
    lo, hi = boot_ci(yt, p); return {"model": name, "accuracy": accuracy_score(yt, p), "macro-F1": f1_score(yt, p, average="macro"), "95% CI": f"{lo:.3f}-{hi:.3f}", "F1 neutral": f1_score(yt, p, average=None)[1]}
trows.append(trow("Majority class (baseline)", models["Majority class (baseline)"].predict(test_df.clean)))
trows.append(trow("TF-IDF + Naive Bayes", models["TF-IDF + Naive Bayes"].predict(test_df.clean)))
trows.append(trow("TF-IDF + LogReg" + (" (DEPLOYED)" if sentiment is sentiment_lr else ""), sentiment_lr.predict_label(test_df.clean)))
if sentiment_bert is not None: trows.append(trow("Fine-tuned DistilBERT" + (" (DEPLOYED)" if sentiment is sentiment_bert else ""), sentiment_bert.predict_label(test_df.clean)))
display(pd.DataFrame(trows).set_index("model").round(3))
pt = sentiment.predict_label(test_df.clean)
print(classification_report(yt, pt, target_names=LABELS, digits=3))
cm = confusion_matrix(yt, pt); plt.figure(figsize=(4, 3.5)); plt.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm): plt.text(j, i, v, ha="center", va="center")
plt.xticks(range(3), LABELS); plt.yticks(range(3), LABELS); plt.xlabel("predicted"); plt.ylabel("actual"); plt.title("Locked test set"); plt.show()


## 6. Issue detector: *what* went wrong?
Six issue types: **D** damaged/defective, **W** wrong item, **A** not as described, **S** fit / compatibility (does the part fit the vehicle?), **L** delivery/shipping, **C** customer service. Two detectors are combined:
- **Keyword rules** (transparent, fast, blind to context) and
- a **zero-shot language model** (`deberta-v3-base-zeroshot`, understands paraphrases, no task training).

For each issue we pick *rules*, *zero-shot* or *rules OR zero-shot* and a threshold, using **leave-one-out F2** on labelled Sets 1+2 (F2 weights recall twice as much as precision: missing a defect costs more than checking an extra order). Where zero-shot is chosen, the rules are added too (union can only add recall). Issues with fewer than 5 tuning examples fall back to rules. **Evaluation is on Set 3: 100 random reviews never used for tuning.**


In [ ]:
r2 = tc.rule_flags(LAB.review); r1 = tc.rule_flags(LAB.review, tc.PATTERNS_V1); z2 = tc.zs_scores(LAB.review.tolist())
for k in ISSUES: LAB[f"r2_{k}"] = r2[k].to_numpy(); LAB[f"r1_{k}"] = r1[k].to_numpy(); LAB[f"z2_{k}"] = z2[k].to_numpy()

THRS = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99]; MIN_POS = 5
def f2(y, p):
    y, p = np.asarray(y, bool), np.asarray(p, bool); tp, fp, fn = (y & p).sum(), (~y & p).sum(), (y & ~p).sum(); d = 5 * tp + 4 * fn + fp
    return 5 * tp / d if d else 0.0
def prf(y, p):
    y, p = np.asarray(y, bool), np.asarray(p, bool); tp, fp, fn = int((y & p).sum()), int((~y & p).sum()), int((y & ~p).sum())
    return (tp / (tp + fp) if tp + fp else 0.0), (tp / (tp + fn) if tp + fn else 0.0), f2(y, p)
predict = lambda m, r, z, t: r if m == "rules" else ((z > t) if m == "zs" else (r | (z > t)))
best_thr = lambda y, r, z, m: max(THRS, key=lambda t: f2(y, predict(m, r, z, t)))
def loo(y, r, z, m):
    if m == "rules": return f2(y, r), None
    out = np.zeros(len(y), bool)
    for i in range(len(y)):
        k = np.arange(len(y)) != i; out[i] = predict(m, r[i:i+1], z[i:i+1], best_thr(y[k], r[k], z[k], m))[0]
    return f2(y, out), best_thr(y, r, z, m)
def choose(d):
    cfg = {}
    for k in ISSUES:
        y, r, z = d[f"y_{k}"].to_numpy(), d[f"r2_{k}"].to_numpy(bool), d[f"z2_{k}"].to_numpy(); n = int(y.sum())
        if n < MIN_POS: cfg[k] = {"method": "rules", "thr": 1.0, "validated": False, "n_pos": n}; continue
        res = {m: loo(y, r, z, m) for m in ["rules", "zs", "either"]}; b = max(res, key=lambda m: res[m][0])
        cfg[k] = {"method": b, "thr": res[b][1] if b != "rules" else 1.0, "validated": True, "n_pos": n, "loo_F2": round(res[b][0], 2)}
    return cfg

tune_df = LAB[LAB.set.isin(["tune", "test"])].reset_index(drop=True); final_df = LAB[LAB.set == "final"].reset_index(drop=True)
CFG = te.harden_config(choose(tune_df)); display(pd.DataFrame(CFG).T)


In [ ]:
def apply(cfg, d): return pd.DataFrame({k: predict(v["method"], d[f"r2_{k}"].to_numpy(bool), d[f"z2_{k}"].to_numpy(), v["thr"]) for k, v in cfg.items()})
flags_final = apply(CFG, final_df); yp = final_df.y_problem.to_numpy(); rows = []
for k in ISSUES:
    y = final_df[f"y_{k}"].to_numpy(); row = {"issue": k, "n_pos (Set 3)": int(y.sum())}
    for name, p in {"v1 rules": final_df[f"r1_{k}"], "v2 rules": final_df[f"r2_{k}"], "zero-shot@0.6": final_df[f"z2_{k}"] > 0.6, "tuned detector": flags_final[k]}.items():
        P, R, F = prf(y, p.to_numpy()); row[name + " P/R"] = f"{P:.2f}/{R:.2f}"
    rows.append(row)
print("Per issue on Set 3 (precision/recall). Where n_pos is small (< 8) treat numbers as indicative only."); display(pd.DataFrame(rows).set_index("issue"))

rng = np.random.default_rng(SEED); dets = {
 "v1 rules": final_df[[f"r1_{k}" for k in ISSUES]].to_numpy().any(1), "v2 rules": final_df[[f"r2_{k}" for k in ISSUES]].to_numpy().any(1),
 "zero-shot@0.6 (any issue)": (final_df[[f"z2_{k}" for k in ISSUES]].to_numpy() > 0.6).any(1), "tuned detector (any issue)": flags_final.to_numpy().any(1)}
rows = []
for n, p in dets.items():
    P, R, F = prf(yp, p); s = [f2(yp[i], p[i]) for i in (rng.integers(0, len(yp), len(yp)) for _ in range(1000))]
    rows.append({"detector": n, "precision": P, "recall": R, "F2": F, "F2 95% CI": "%.2f-%.2f" % tuple(np.percentile(s, [2.5, 97.5])), "missed": int((yp & ~p).sum()), "false alarms": int((~yp & p).sum())})
print(f"\nProblem-order detection on Set 3 ({int(yp.sum())} real problems of {len(yp)} random reviews):"); display(pd.DataFrame(rows).set_index("detector").round(2))
p = dets["tuned detector (any issue)"]
print("MISSED examples:"); [print(f"  [{r.id}] {r.issue_codes} | {r.review[:120]}") for _, r in final_df[yp & ~p].head(6).iterrows()]
print("FALSE ALARMS:"); [print(f"  [{r.id}] | {r.review[:120]}") for _, r in final_df[~yp & p].head(6).iterrows()]


## 6b. Which part is the review about?
Knowing the *problem* is not enough for a returns team: they also need the *part*. A review about tyres goes to a different person than one about wiper blades. `triage_core.mentioned_parts()` names the part a review talks about, from a transparent keyword lexicon of about 70 auto-part names grouped like the retailer's catalogue (tyres & wheels, brakes, lights, wipers, floor mats, filters, batteries, tools, oils & fluids ...). A specific entry wins over a general one ("tire pressure gauge" is a gauge, "check engine light" is not a bulb). It is a lexicon, **not a trained model**, and there is no ground truth for the part, so we report how often it names a part and show examples to inspect. In an informal hand check of 40 random matches (by the AI assistant), about 85% named a sensible part; the mistakes were mostly reviews that mention another object in passing.


In [ ]:
pool_txt = ad.tidy(stream_df.iloc[300_000:420_000]); pool_txt = pool_txt[pool_txt.text.str.len().between(40, 400)].sample(3000, random_state=11).reset_index(drop=True)
named = [tc.mentioned_parts(t) for t in pool_txt.text]
print(f"{np.mean([len(x) > 0 for x in named]):.0%} of {len(named)} random automotive reviews name a part from the lexicon (the rest say things like 'fits perfectly' without naming one).")
print("\nFirst-named part, by catalogue group:"); print(pd.Series([x[0]["group"] for x in named if x]).value_counts().to_string())
print("\nMost common parts:", pd.Series([x[0]["name"] for x in named if x]).value_counts().head(10).to_dict())
print("\n12 random examples to inspect (part <- matched word | review):")
for i in np.random.default_rng(5).choice([i for i, x in enumerate(named) if x], 12, replace=False):
    print(f"  [{', '.join(p['name'] + ' <- ' + p['phrase'] for p in named[i])}] | {pool_txt.text[i][:95]}")


## 7. Image evidence: damaged item? wrong item?
- **Damage:** three detectors compared on a held-out half: (a) *zero-shot CLIP* with text prompts, (b) a *trained classifier* (logistic regression on frozen CLIP embeddings, the "embedding + simple classifier" recipe from the course's ViT lab), (c) *distance from the good reference photos of the same product* (the standard anomaly-detection idea; a retailer has catalogue photos). The best by evidence is used downstream. *Earlier finding: text-prompt zero-shot was near chance on subtle defects (AUC 0.57) and gave about 30% false wrong-item alarms, which is why reference photos were added.*
- **Wrong item (misdelivery):** is the customer's photo closest to the **ordered product's** reference photos (compared with CLIP text prompts)? A low match flags a possible wrong delivery.
- **Image risk** = max(P(damaged), 1 - P(match)) feeds the priority score.

**Data limits, stated plainly:** [MVTec AD](https://www.mvtec.com/company/research/datasets/mvtec-ad) (real good vs defective images of industrial objects; CC BY-NC-SA, coursework use) via the `TheoM55/mvtec_all_objects_split` mirror. These are **factory-style close-ups, not customer snapshots**: they validate the *method* on real labelled defects, and the retailer would re-test on its own return photos. We use 7 object types that map onto an auto-parts catalogue (screw and metal nut = fasteners, cable = wiring, bottle = fluids, transistor = electronic modules, leather = seat covers, carpet = floor mats). No public dataset pairs real reviews with labelled damage photos, so the text+photo demos are **staged**.


In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download
from sklearn.model_selection import train_test_split
from triage_image import ClipScorer, image_risk
REPO = "TheoM55/mvtec_all_objects_split"
# Seven MVTec AD object types that map onto an auto-parts catalogue (friendly names used by the app: triage_ui.PRODUCT_LABELS)
CATS = ["screw", "metal_nut", "cable", "bottle", "transistor", "leather", "carpet"]
repo_files = list_repo_files(REPO, repo_type="dataset"); frames = []
for c in CATS:
    for sp in ("test", "train"):      # test = good + defective photos; train = extra GOOD photos (most photos a shop receives are fine)
        for fn in [f for f in repo_files if f.startswith(f"data/{c}.{sp}")]: frames.append(pd.read_parquet(hf_hub_download(REPO, fn, repo_type="dataset")))
raw_img = pd.concat(frames, ignore_index=True)
def to_pil(v):
    if isinstance(v, dict) and v.get("bytes") is not None: im = Image.open(io.BytesIO(v["bytes"])).convert("RGB")
    elif isinstance(v, (bytes, bytearray)): im = Image.open(io.BytesIO(v)).convert("RGB")
    else: im = Image.open(v).convert("RGB")
    im.thumbnail((448, 448)); return im          # CLIP looks at 224 px; smaller images keep memory low
icol = "image_path" if "image_path" in raw_img.columns else [c for c in raw_img.columns if "image" in c][0]
img_df = pd.DataFrame({"obj": raw_img.object.str.replace("_", " "), "defect": raw_img.defect.astype(str)}); img_df["is_defect"] = img_df.defect.str.lower() != "good"
img_df["img"] = [to_pil(v) for v in raw_img[icol]]; PRODUCTS = sorted(img_df.obj.unique())
clip = ClipScorer(); emb = clip.embed_images(img_df.img.tolist()); yi = img_df.is_defect.to_numpy(); prods = img_df.obj.tolist()
display(img_df.groupby("obj").is_defect.agg(["size", "sum"]).rename(columns={"size": "images", "sum": "defective"}))


In [ ]:
import triage_image as ti
strat = img_df.obj + "_" + img_df.is_defect.astype(str)
itr, ite = train_test_split(np.arange(len(img_df)), test_size=0.5, random_state=SEED, stratify=strat)
Etr, Ete = emb[itr], emb[ite]; ptr, pte = np.array(prods)[itr], np.array(prods)[ite]; ytr_i, yte_i = yi[itr], yi[ite]
# Reference photos = what a retailer holds in its catalogue: GOOD photos of each product (train half only)
refs = {o: Etr[(ptr == o) & (~ytr_i)] for o in PRODUCTS}; protos = ti.make_prototypes(refs)

# ---- Damage: three detectors on the SAME held-out half ----
p_zero = clip.defect_prob(emb, prods)                                              # (a) zero-shot CLIP, text prompts
lr_img = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced").fit(Etr, ytr_i); p_lr = lr_img.predict_proba(emb)[:, 1]   # (b) trained
a_full = np.zeros(len(img_df)); a_full[itr] = ti.anomaly_scores(Etr, ptr, refs, exclude_self=True); a_full[ite] = ti.anomaly_scores(Ete, pte, refs)   # (c) distance from good reference photos
platt = LogisticRegression(class_weight="balanced").fit(a_full[itr].reshape(-1, 1), ytr_i); p_ref = platt.predict_proba(a_full.reshape(-1, 1))[:, 1]
def aucs(score):
    per = [roc_auc_score(yte_i[pte == o], score[ite][pte == o]) for o in PRODUCTS]
    return {"pooled AUC": roc_auc_score(yte_i, score[ite]), "mean per-product AUC": float(np.mean(per)), "worst product": float(np.min(per))}
dm = pd.DataFrame({"(a) zero-shot CLIP (text)": aucs(p_zero), "(b) trained LogReg on embeddings": aucs(p_lr), "(c) distance from good reference photos": aucs(a_full)}).T
print(f"Damage detection, held-out half: {len(ite)} images, {int(yte_i.sum())} defective ({yte_i.mean():.0%}: precision depends on this rate, so we use AUC)"); display(dm.round(2))
rows = []
for o in PRODUCTS:   # (b) on products it never saw in training
    m = (img_df.obj == o).to_numpy(); c = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced").fit(emb[~m], yi[~m])
    rows.append({"unseen product": o, "(b) trained AUC": roc_auc_score(yi[m], c.predict_proba(emb[m])[:, 1])})
print("(b) trained classifier on a product type it has never seen (leave-one-product-out):"); display(pd.DataFrame(rows).set_index("unseen product").round(2).T)
best_damage = dm["mean per-product AUC"].idxmax(); print("damage model used downstream:", best_damage)
amap = {"(a)": "zero", "(b)": "lr", "(c)": "ref"}[best_damage[:3]]

def damage_prob(e, prods_):
    prods_ = list(prods_)
    if amap == "lr": return lr_img.predict_proba(e)[:, 1]
    if amap == "ref" and all(p in refs for p in prods_): return platt.predict_proba(ti.anomaly_scores(e, prods_, refs).reshape(-1, 1))[:, 1]
    return clip.defect_prob(e, prods_)
def match_prob(e, prods_):
    prods_ = list(prods_)
    return ti.ref_match_prob(e, prods_, protos) if all(p in protos for p in prods_) else np.ones(len(prods_))   # unknown product: no mismatch evidence

# ---- Wrong item: is the photo the ORDERED product? text-based vs reference-based, held-out half ----
rng = np.random.default_rng(SEED); bad = np.array([rng.choice([p for p in PRODUCTS if p != o]) for o in pte])
res = {}
for name, f in {"CLIP text prompts (6-way)": lambda o: clip.match_prob(Ete, o, PRODUCTS), "reference photos (catalogue)": lambda o: ti.ref_match_prob(Ete, o, protos)}.items():
    ok, nok = f(pte), f(bad); sc = np.r_[ok, nok]; lab_m = np.r_[np.ones(len(ok)), np.zeros(len(nok))]
    res[name] = {"AUC": roc_auc_score(lab_m, sc), "flag precision": precision_score(lab_m == 0, sc < 0.5), "flag recall": recall_score(lab_m == 0, sc < 0.5), "false alarms on correct photos": float((ok < 0.5).mean())}
print("Wrong-item detection (matched vs mismatched order; flag when P(match) < 0.5):"); display(pd.DataFrame(res).T.round(3))
print("Caveat: 7 very different products; real catalogues contain look-alike items, so expect lower numbers. Reference photos here come from the same dataset.")


### 7a. When should a photo count as "damaged"?
A fixed probability cut-off does not work across part types: a leather seat cover with a tear is easy to spot (AUC about 1.0), a screw with a tiny defect is not. So the cut-off is chosen **per part type, on the training half only**, using the raw anomaly score (distance from the nearest good reference photo): the lowest cut-off that keeps precision at 0.90 or better. If no cut-off reaches that target, **the damage check is switched off for that part type** and the app says so; the wrong-part check still works. Below the cut-off a photo adds no risk to the order's priority. We report what the cut-offs do on the held-out half.


In [ ]:
a_tr = a_full[itr]          # anomaly scores of the training half, computed above without comparing a reference photo with itself
def pick_cut(a, y, target=0.90, min_flag=3):
    for t in sorted(np.unique(np.round(a[y], 4))):
        f = a >= t; tp = int((f & y).sum()); fp = int((f & ~y).sum())
        if tp >= min_flag and tp / max(tp + fp, 1) >= target: return float(t)
    return None             # no cut-off reaches the target: damage check OFF for this part type
ANOM_THR = {o: pick_cut(a_tr[ptr == o], ytr_i[ptr == o]) for o in PRODUCTS}
a_te = a_full[ite]; rows = []
for o in PRODUCTS:
    m = pte == o; t = ANOM_THR[o]; y_ = yte_i[m]; auc_ = round(roc_auc_score(y_, a_te[m]), 2)
    if t is None: rows.append({"part": o, "cut-off": "OFF", "defective (held-out)": int(y_.sum()), "flagged": 0, "precision": np.nan, "recall": 0.0, "AUC": auc_}); continue
    f = a_te[m] >= t
    rows.append({"part": o, "cut-off": round(t, 4), "defective (held-out)": int(y_.sum()), "flagged": int(f.sum()), "precision": round(precision_score(y_, f, zero_division=0), 2), "recall": round(recall_score(y_, f), 2), "AUC": auc_})
display(pd.DataFrame(rows).set_index("part"))
f_all = np.array([(ANOM_THR[o] is not None) and (a >= ANOM_THR[o]) for o, a in zip(pte, a_te)])
DMG_P, DMG_R = precision_score(yte_i, f_all, zero_division=0), recall_score(yte_i, f_all)
print(f"All parts, held-out half (cut-offs chosen on the training half): damage flag precision {DMG_P:.2f}, recall {DMG_R:.2f}  ({int(f_all.sum())} flagged of {int(yte_i.sum())} defective photos).")
print("Damage check OFF for:", [o for o in PRODUCTS if ANOM_THR[o] is None] or "none")
ev_final = ti.ImageEvidence(clip, refs, protos, platt, lr_img, amap, PRODUCTS, damage_thr=0.5, anom_thr=ANOM_THR, known_margin=0.08)   # the object the app uses
damage_prob = lambda e, p: ev_final.damage_prob(e, p); match_prob = lambda e, p: ev_final.match_prob(e, p)
risk_fn = lambda e, p: ev_final.risk_from_emb(e, p)[0]


### 7b. A failure found in testing: photos of something that is not in the catalogue
Comparing a photo with only 6 known products forces a choice, so a photo of a completely different object can still "match" the nearest product (a tester uploaded an aircraft cockpit as a "bottle" and got an 83% match). The fix is an **absolute** check as well: the photo must look as similar to the ordered product as a normal good photo does. We test it on **three product types the catalogue does not contain** (toothbrush, zipper, wood) and check it does not create false alarms on correct-product photos (good or defective). Testing also showed that the first version of the absolute check labelled **damaged** parts as a wrong item (a badly damaged part looks less like a normal photo), so the final rule asks instead: does the photo look like *some* known part, or is it clearly closest to the ordered part?


In [ ]:
extra = ["toothbrush", "zipper", "wood", "pill", "hazelnut", "capsule", "grid", "tile"]; fr = []
for c in extra:
    for fn in [f for f in repo_files if f.startswith(f"data/{c}.test")]: fr.append(pd.read_parquet(hf_hub_download(REPO, fn, repo_type="dataset")))
ood = pd.concat(fr, ignore_index=True); E_ood = clip.embed_images([to_pil(v) for v in ood[icol]])
ev_tmp = ev_final
rng = np.random.default_rng(SEED); ordered = list(rng.choice(PRODUCTS, len(E_ood)))
old_m, v1_m, new_m = ti.ref_match_prob(E_ood, ordered, protos), ev_tmp.match_prob_v1(E_ood, ordered), ev_tmp.match_prob(E_ood, ordered)
ok_old, ok_v1, ok_new = ti.ref_match_prob(Ete, list(pte), protos), ev_tmp.match_prob_v1(Ete, list(pte)), ev_tmp.match_prob(Ete, list(pte))
print(f"Photos of {len(E_ood)} items NOT in the catalogue, wrongly accepted as the ordered product (match >= 0.5):")
print(f"  relative check only: {(old_m >= 0.5).mean():.0%}   |   + similarity floor (first version): {(v1_m >= 0.5).mean():.0%}   |   + known-part check (final): {(new_m >= 0.5).mean():.0%}")
print(f"False 'wrong item' alarms on correct-part photos (held-out half, {len(Ete)} photos), good photos / DEFECTIVE photos:")
print(f"  relative only: {(ok_old[~yte_i] < 0.5).mean():.1%} / {(ok_old[yte_i] < 0.5).mean():.1%}   |   first version: {(ok_v1[~yte_i] < 0.5).mean():.1%} / {(ok_v1[yte_i] < 0.5).mean():.1%}   |   final: {(ok_new[~yte_i] < 0.5).mean():.1%} / {(ok_new[yte_i] < 0.5).mean():.1%}")
print("The first version mistook some damaged parts for a wrong item; the final rule does not.")


### 7c. Course baseline: a CNN trained from scratch (Keras)
The course's image labs train a small **convolutional neural network** in Keras. We run the same recipe (3 conv blocks with pooling and dropout, then a dense layer) on the same defect photos and the same train/held-out split, to compare it with the pretrained-embedding approach above. With only about 1,300 training images (roughly 300 of them defective), a network trained from scratch has little to learn from, which is exactly the case for **transfer learning** (reusing CLIP's pretrained features).


In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
    tf.random.set_seed(SEED)
    X_all = np.stack([np.array(im.resize((64, 64))) for im in img_df.img]).astype("float32") / 255
    cnn = Sequential([Input((64, 64, 3)),
                      Conv2D(32, (3, 3), activation="relu"), MaxPooling2D(2), Dropout(0.1),
                      Conv2D(64, (3, 3), activation="relu"), MaxPooling2D(2), Dropout(0.2),
                      Conv2D(64, (3, 3), activation="relu"), MaxPooling2D(2), Dropout(0.2),
                      Flatten(), Dense(128, activation="relu"), Dropout(0.2), Dense(1, activation="sigmoid")])
    cnn.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
    cw = {0: len(ytr_i) / (2 * (~ytr_i).sum()), 1: len(ytr_i) / (2 * ytr_i.sum())}          # balance defective / good
    cnn.fit(X_all[itr], ytr_i.astype("float32"), epochs=20, batch_size=16, class_weight=cw, verbose=0)
    p_cnn = cnn.predict(X_all[ite], verbose=0).ravel()
    per = [roc_auc_score(yte_i[pte == o], p_cnn[pte == o]) for o in PRODUCTS]
    cnn_row = {"pooled AUC": roc_auc_score(yte_i, p_cnn), "mean per-product AUC": float(np.mean(per)), "worst product": float(np.min(per))}
    cmp = pd.concat([dm, pd.DataFrame({"(d) Keras CNN from scratch (course recipe)": cnn_row}).T]); display(cmp.round(2))
    print(f"A CNN trained from scratch on {len(itr)} photos versus CLIP embeddings: transfer learning is the better fit for small image datasets.")
except Exception as e:
    print("Keras CNN baseline skipped:", e)


## 8. The triage engine, evaluated on Set 3 (100 random, unseen reviews)
**Priority score** = issue risk x (0.6 + 0.4 x P(negative)). Issue risk combines the detected issues' severity weights (damaged/wrong item 0.9, delivery/service 0.7, not as described 0.6, size 0.4; **assumptions** about relative cost, tested for sensitivity below) with the photo's risk (weight 0.8) when available. A negative tone raises urgency, but a calm review with a real issue is still surfaced. In the app, a happy-sounding review that contains a confirmed complaint after a contrast word ("looks great, *but* it doesn't fit my Civic") is flagged as a mixed complaint: measured on 3,000 five-star reviews this touches 2.0% of them, against 8.8% if every confirmed phrase were flagged (most of those are negated or irrelevant mentions, such as "the paint never chipped").
We compare the ranking with what a shop does today (**sort by star rating**), with **sentiment only**, and with **issues only** (ablation).


In [ ]:
queue = te.triage(final_df.review, sentiment, flags=flags_final, sort=False)
y = final_df.y_problem.to_numpy(); p_neg = queue.p_negative.to_numpy(); prio = queue.priority_score.to_numpy(); stars = final_df.stars.astype(int).to_numpy()
flagged_lists = [[k for k in ISSUES if flags_final.loc[i, k]] for i in range(len(y))]
cands = {"Priority score (ours)": prio, "Issues only (tone ignored)": np.array([te.priority_score(0.5, flagged_lists[i]) for i in range(len(y))]),
         "Sentiment only: P(negative)": p_neg, "Star rating (lowest first)": -stars.astype(float)}
p_neg_lr = sentiment_lr.predict_proba(final_df.review.tolist())[:, 0]
if sentiment is not sentiment_lr: cands["Priority score with TF-IDF sentiment"] = np.array([te.priority_score(p_neg_lr[i], flagged_lists[i]) for i in range(len(y))])
rr = np.random.default_rng(1); boots = [b for b in (rr.integers(0, len(y), len(y)) for _ in range(2000)) if 0 < y[b].sum() < len(b)]
base = cands["Priority score (ours)"]; rows = []
for n, s in cands.items():
    a = np.array([roc_auc_score(y[b], s[b]) for b in boots]); d = np.array([roc_auc_score(y[b], base[b]) - roc_auc_score(y[b], s[b]) for b in boots])
    rows.append({"ranking": n, "ROC-AUC": round(roc_auc_score(y, s), 3), "95% CI": "%.2f-%.2f" % tuple(np.percentile(a, [2.5, 97.5])),
                 "gain of ours": "-" if n.startswith("Priority") else "%.3f (%.3f to %.3f)" % (d.mean(), *np.percentile(d, [2.5, 97.5]))})
display(pd.DataFrame(rows).set_index("ranking"))
tie = lambda s: s + np.random.default_rng(0).uniform(0, 1e-9, len(s)); x = np.linspace(0, 1, len(y) + 1)
plt.figure(figsize=(6, 4.5))
for n, s in cands.items(): plt.plot(x, np.r_[0, np.cumsum(y[np.argsort(-tie(s))]) / y.sum()], label=n)
plt.plot(x, np.minimum(1, x * len(y) / y.sum()), "--", color="grey", label="ideal"); plt.plot([0, 1], [0, 1], ":", color="grey", label="random")
plt.xlabel("share of reviews checked (top of queue first)"); plt.ylabel("share of real problem orders found"); plt.legend(); plt.title("Gain chart, Set 3 (random reviews)"); plt.show()


In [ ]:
print(f"Set 3: {int(y.sum())} real problem orders in {len(y)} random reviews; hidden behind 4-5 stars: {int((y & (stars >= 4)).sum())}; behind P(negative) < 0.5: {int((y & (p_neg < 0.5)).sum())}")
print("Share of problem orders by star rating:", {s: f"{int(y[stars == s].sum())}/{int((stars == s).sum())}" for s in range(1, 6)})
display(pd.crosstab(queue.priority, np.where(y, "real problem", "no problem")).reindex(["HIGH", "MEDIUM", "LOW"]).fillna(0).astype(int))
aucs = []; rs = np.random.default_rng(0)
for _ in range(200):
    sev = {k: float(np.clip(v * rs.uniform(0.7, 1.3), 0.05, 0.99)) for k, v in te.SEVERITY.items()}
    aucs.append(roc_auc_score(y, [te.priority_score(p_neg[i], flagged_lists[i], severity=sev) for i in range(len(y))]))
print(f"Sensitivity to severity weights (+/-30%): AUC mean {np.mean(aucs):.3f}, min {np.min(aucs):.3f}, max {np.max(aucs):.3f} (default {roc_auc_score(y, prio):.3f})")


## 8b. Honest check: does the hand-built score beat simple baselines, and can a *learned* combination do better?
On random reviews the hand-built priority score is **not clearly better** than sentiment alone or star rating (see the table above: the confidence intervals overlap). One reason is that it uses hand-chosen weights. Here we let a small **logistic regression learn how much to trust each signal** (calibrated sentiment, six zero-shot issue scores, six rule flags; optionally the star rating) from the 160 hand-checked reviews of Sets 1+2, and test it on **Set 3** (never used for training or tuning). Stars are shown as an optional input because real shops have them.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
def feats(d, with_stars=False):
    P = sentiment.predict_proba(d.review.tolist())
    cols = [P[:, 0], P[:, 1]] + [d[f"z2_{k}"].to_numpy() for k in ISSUES] + [d[f"r2_{k}"].to_numpy().astype(float) for k in ISSUES]
    if with_stars: cols.append(d.stars.astype(float).to_numpy())
    return np.column_stack(cols)
ytr = tune_df.y_problem.to_numpy(); fusion_scores = {}; fusion_models = {}
for name, ws in [("Learned fusion: text only", False), ("Learned fusion: text + stars", True)]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=2000)).fit(feats(tune_df, ws), ytr)
    fusion_models[name] = (m, ws); fusion_scores[name] = m.predict_proba(feats(final_df, ws))[:, 1]
issue_risk = np.array([1 - np.prod([1 - te.SEVERITY[k] for k in fl]) if fl else 0.0 for fl in flagged_lists])
cmp_scores = {"Sentiment only (deployed model)": p_neg, "Star rating (lowest first)": -stars.astype(float), "Hand-built priority score": prio,
              "Simple blend: 0.7 x P(neg) + 0.3 x issue risk": 0.7 * p_neg + 0.3 * issue_risk, **fusion_scores}
ref_s, ref_st = cmp_scores["Sentiment only (deployed model)"], cmp_scores["Star rating (lowest first)"]; rows = []
for n, s in cmp_scores.items():
    a = np.array([roc_auc_score(y[b], s[b]) for b in boots])
    d1 = np.array([roc_auc_score(y[b], s[b]) - roc_auc_score(y[b], ref_s[b]) for b in boots]); d2 = np.array([roc_auc_score(y[b], s[b]) - roc_auc_score(y[b], ref_st[b]) for b in boots])
    rows.append({"ranking": n, "ROC-AUC": round(roc_auc_score(y, s), 3), "95% CI": "%.2f-%.2f" % tuple(np.percentile(a, [2.5, 97.5])),
                 "vs sentiment only": "%+.3f (%.3f to %.3f)" % (d1.mean(), *np.percentile(d1, [2.5, 97.5])),
                 "vs star rating": "%+.3f (%.3f to %.3f)" % (d2.mean(), *np.percentile(d2, [2.5, 97.5]))})
display(pd.DataFrame(rows).set_index("ranking"))


## 8c. What stars and sentiment can never do: say *what* went wrong
A star rating or sentiment score gives a ranking but no **issue type**, so a support team still has to read every flagged review to route it. Among the real problem orders, how often does our detector name a correct issue type?


In [ ]:
true_sets = [set(c for c in s.split(",") if c) for s in final_df.issue_codes]
pred_sets = [set(CODES[k] for k in ISSUES if flags_final.loc[i, k]) for i in range(len(y))]
idx = np.where(y)[0]; flagged_real = [i for i in idx if pred_sets[i]]
hit = [i for i in flagged_real if true_sets[i] & pred_sets[i]]; exact = [i for i in flagged_real if true_sets[i] == pred_sets[i]]
print(f"Real problem orders: {len(idx)} | flagged with at least one issue type: {len(flagged_real)} | at least one CORRECT type: {len(hit)} ({len(hit) / len(idx):.0%} of all real problems) | exact issue set: {len(exact)}")
print("Star rating / sentiment routing accuracy: 0 by construction (they carry no issue type).")
by_issue = pd.DataFrame({CODES[k]: {"real": int(final_df[f"y_{k}"].sum()), "flagged": int(flags_final[k].sum()), "correct flags": int((final_df[f"y_{k}"].to_numpy() & flags_final[k].to_numpy()).sum())} for k in ISSUES}).T
display(by_issue)


## 8d. The brief's third goal: combine the review and the photo to flag problematic orders
No public dataset pairs real reviews with labelled customer photos, so we test the **combination logic** on **simulated orders**: each order takes a *real unseen review* (with its human label) and a *real held-out photo*, paired at random. An order is a real problem if the review reports one, **or** the photo is damaged, **or** the photo shows a different product from the one ordered (15% of orders are wrong-item, 20% damaged, the rest fine). We then compare three flags: review only, photo only, and the combination (the priority score with the photo's risk added).
**Read this carefully:** because the problem is defined by either source, a combination *should* beat each one alone; the test checks that our fusion achieves that with real model outputs, not that it predicts real customer behaviour.


In [ ]:
rng = np.random.default_rng(SEED); N_SIM = 300
good_i = [i for i in ite if not img_df.is_defect[i]]; bad_i = [i for i in ite if img_df.is_defect[i]]
sim = []
for _ in range(N_SIM):
    ri = int(rng.integers(len(final_df))); ordered = str(rng.choice(PRODUCTS)); u = rng.random()
    if u < 0.15:   kind, pi_ = "wrong item", int(rng.choice([i for i in ite if img_df.obj[i] != ordered]))
    elif u < 0.35: kind, pi_ = "damaged", int(rng.choice([i for i in bad_i if img_df.obj[i] == ordered]))
    else:          kind, pi_ = "fine", int(rng.choice([i for i in good_i if img_df.obj[i] == ordered]))
    sim.append((ri, ordered, kind, pi_))
ri_ = np.array([s[0] for s in sim]); kinds = np.array([s[2] for s in sim]); ord_s = [s[1] for s in sim]
risk_s = ev_final.risk_from_emb(emb[[s[3] for s in sim]], ord_s)[0]
truth = y[ri_] | (kinds != "fine")
text_flag = queue.priority.to_numpy()[ri_] != "LOW"
photo_flag_ = risk_s >= 0.5
comb_flag = np.array([te.bucket(te.priority_score(p_neg[r], flagged_lists[r], image_risk=float(rk))) in ("HIGH", "MEDIUM") or float(rk) >= te.PHOTO_FLAG for r, rk in zip(ri_, risk_s)])
res8 = pd.DataFrame([{"flag based on": n, "precision": precision_score(truth, f), "recall": recall_score(truth, f), "F1": f1_score(truth, f)}
                     for n, f in {"review only": text_flag, "photo only": photo_flag_, "review + photo (combined)": comb_flag}.items()]).set_index("flag based on")
display(res8.round(2))
print(f"{N_SIM} simulated orders, {int(truth.sum())} real problems ({int((kinds == 'wrong item').sum())} wrong-item and {int((kinds == 'damaged').sum())} damaged photos). Pairing is random: this tests the combination logic, not real customer behaviour.")
print("Problems the review alone misses but the photo finds:", int((truth & ~text_flag & comb_flag).sum()), "| false alarms added by the photo:", int((~truth & comb_flag & ~text_flag).sum()))


## 9. Business value: how much work does the queue save?
A shop today checks reviews by **star rating** (e.g. "look at everything with 3 stars or fewer"). We compare that workload with our queue on the same 100 random reviews: what share of reviews must be checked, how many real problems are caught, how many checks are wasted.
**Cost model (assumptions, editable):** each check costs `C_CHECK`, each missed problem costs `C_MISS` (refund, churn, reputation). The defaults are illustrative, so we show how the ranking of strategies changes as `C_MISS` changes.


In [ ]:
N = len(y); strategies = {"Stars <= 2": stars <= 2, "Stars <= 3": stars <= 3, "Stars <= 4": stars <= 4,
                         "Our queue: HIGH only": (queue.priority == "HIGH").to_numpy(), "Our queue: HIGH + MEDIUM": (queue.priority != "LOW").to_numpy()}
rows = [{"strategy": n, "reviews checked": f"{f.mean():.0%}", "real problems caught": f"{(y & f).sum() / y.sum():.0%}",
         "wasted checks / 100": round(100 * (f & ~y).sum() / N, 1), "missed / 100": round(100 * (y & ~f).sum() / N, 1)} for n, f in strategies.items()]
display(pd.DataFrame(rows).set_index("strategy"))
C_CHECK = 1.5          # assumed cost of one agent check (arbitrary currency units)
sens = {n: [round(((f.sum() * C_CHECK + (y & ~f).sum() * cm) / N) * 1000) for cm in (5, 10, 25, 50, 100)] for n, f in strategies.items()}
print("Expected cost per 1,000 reviews, by cost of a MISSED problem (C_MISS):"); display(pd.DataFrame(sens, index=["C_MISS=5", "10", "25", "50", "100"]).T)
print("Note: this dataset is balanced by star rating, so problems are more common than in a real shop; the rankings above, not the absolute costs, are the transferable result.")


### Staged text + photo demo
Reviews paired with MVTec photos (no public dataset pairs real reviews with labelled damage photos, so these pairs are **staged**). Notice how a happy review with a damaged photo, or a happy review with the wrong item pictured, is raised by the image evidence.


In [ ]:
pick = lambda obj, d: [i for i in ite if img_df.obj[i] == obj and img_df.is_defect[i] == d][0]   # photos from the held-out half (never used as references)
orders = [("Very disappointed, it arrived broken and nobody answers my emails.", "bottle", pick("bottle", True)),
          ("Love it, works great and arrived fast!", "leather", pick("leather", True)), ("Love it, works great and arrived fast!", "cable", pick("cable", False)),
          ("This is not what I ordered, I want my money back.", "screw", pick("transistor", False)), ("It is okay, does the job.", "metal nut", pick("metal nut", False)),
          ("Great quality, thank you!", "screw", pick("carpet", False))]
tx = [o[0] for o in orders]; pr = [o[1] for o in orders]; ids = [o[2] for o in orders]; e = emb[ids]
risk = risk_fn(e, pr)
a = te.triage(tx, sentiment, CFG, image_risk=risk, sort=False); b = te.triage(tx, sentiment, CFG, sort=False)
display(pd.DataFrame({"review": [t[:50] for t in tx], "ordered": pr, "photo shows": [img_df.obj[i] + (" (defective)" if img_df.is_defect[i] else "") for i in ids],
                      "image_risk": risk.round(2), "priority (text only)": b.priority.values, "priority (text + photo)": a.priority.values}))
fig, ax = plt.subplots(1, len(ids), figsize=(2.6 * len(ids), 2.8))
for ax_, i, k in zip(ax, ids, range(len(ids))): ax_.imshow(img_df.img[i]); ax_.axis("off"); ax_.set_title(f"order {k}", fontsize=9)
plt.show()


## 9b. From reviews to decisions: the insight layer for managers
A ranked list helps a support agent. Managers who set strategy and plans (customer experience, product, quality, logistics) need to know **where problems concentrate, whether they are rising, what to fix first, who owns it, and roughly what it costs**. `triage_insights.py` turns the triaged reviews into exactly that.

It runs here on a **demo dataset** built from real Amazon Automotive reviews that were **not** used for training or labelling: real review text, real star ratings, real dates and real product identifiers (the ten most-reviewed products in that slice, about 30 reviews each, natural star mix; product names are looked up from Amazon's product catalogue file). Nothing is invented or planted, so any hotspot it shows is whatever the real data contains, but with about 30 reviews per product the product comparison is a demonstration, not a finding. Costs use **illustrative assumptions** (average order value $40; share of problem reviews that end in a refund per issue type).

Design choices that make the output trustworthy: a problem is **confirmed** only when a matching phrase is found in the review; issues suggested by the language model alone are only **possible** (they count half in the cost and impact estimates, and a review cannot be "Act today" on possible issues alone).


In [ ]:
import triage_insights as ins
demo_raw = pd.read_csv("demo_auto_reviews.csv"); dd = ins.standardise(demo_raw)
qd = ins.analyse(dd, sentiment, CFG); gcol = ins.group_column(qd); AOV = 40.0
print(f"{len(qd)} reviews analysed; grouping column: {gcol}\n")
for line in ins.narrative(qd, AOV, gcol): print("-", line)
it_ = ins.issue_table(qd, AOV); display(it_[["label", "confirmed", "possible", "reviews", "share", "est_cost"]].round(2))
hot_ = ins.hotspots(qd, gcol); display(hot_.round(2))
print("\nRecommended priorities:")
for r in ins.recommendations(qd, AOV, 3, gcol):
    print(f"\n{r['rank']}. {r['label']}: {r['reviews']} reviews ({r['confirmed']} confirmed, {r['possible']} to check), about ${r['est_cost']:,.0f} at risk. Owner: {r['owner']}")
    if r["hotspot"]: print(f"   Concentrated in {r['hotspot']['name']}: {r['hotspot']['n']} of {r['hotspot']['of']} reviews on this issue")
    for a in r["actions"]: print("   -", a)
for fig_ in (ins.plot_issues(it_), ins.plot_hotspots(hot_, gcol), ins.plot_trend(ins.trend(qd))): display(fig_)
print("Excel report written:", ins.excel_report("demo_report.xlsx", qd, AOV))


## 9c. Sanity tests for the reusable code
Good engineering means the reusable pieces are checked automatically. These assertions verify the behaviours the design depends on (they stop the notebook if one fails).

**Data model.** Everything works on one *review record*: `review` text (required) plus optional `product` (or `category`/`supplier`), `date`, `rating`, `order_id` and `photo`. `triage_insights.standardise()` maps a team's own column names to this schema, and every output keeps the team's columns beside the results.


In [ ]:
def _t(name, cond):
    print(("PASS  " if cond else "FAIL  ") + name); assert cond, name
_t("clean() lowercases and strips html and urls", tc.clean("Great <b>PRODUCT</b> http://x.com  ") == "great product")
rf = tc.rule_flags(["The box arrived crushed and the lamp is cracked", "Love it, fits perfectly"])
_t("keyword rules flag damage and stay quiet on praise", bool(rf.loc[0, "damaged_defective"]) and not rf.loc[1].any())
_t("priority rises with more issues", te.priority_score(0.5, ["damaged_defective", "delivery_shipping"]) > te.priority_score(0.5, ["damaged_defective"]))
_t("priority rises with a more negative tone", te.priority_score(0.9, ["size_fit"]) > te.priority_score(0.1, ["size_fit"]))
_t("negative tone without any issue stays low", te.bucket(te.priority_score(0.95, [])) == "LOW")
_t("harden_config adds the rules wherever only zero-shot was chosen", te.harden_config({"x": {"method": "zs", "thr": 0.5}})["x"]["method"] == "either")
_t("calibrated probabilities sum to 1", abs(float(sentiment.predict_proba(["ok"]).sum()) - 1) < 1e-6)
q_t = te.triage(["bad", "The box was crushed and cracked, very disappointed"], sentiment, CFG).set_index("review")
_t("a review whose issue is only 'possible' is never HIGH", q_t.loc["bad", "priority"] != "HIGH")
_t("standardise() maps column names", list(ins.standardise(pd.DataFrame({"Comment": ["x"], "Item": ["p"]})).columns) == ["review", "product"])
print("\nall sanity tests passed")


## 10. Package the system for the business team
The trained system is exported so a **standalone web app** (`app.py`: a work-queue dashboard for customer-experience teams) can load it without re-training. This cell writes an `artifacts/` folder (sentiment model, issue-detector settings, image evidence, example photos, sample reviews, headline metrics) and downloads it as `triage_artifacts.zip`. See the repository README for how to run the app (Colab, locally, or Hugging Face Spaces).


In [ ]:
import shutil
ART = "artifacts"; shutil.rmtree(ART, ignore_errors=True); os.makedirs(ART + "/examples"); os.makedirs(ART + "/image")
te.save_sentiment(sentiment, f"{ART}/sentiment")
json.dump(CFG, open(f"{ART}/issue_config.json", "w"), indent=2)
ev = ev_final; ev.save(f"{ART}/image/image_evidence.joblib")
ex = {}
for o in PRODUCTS:                       # one good photo per part, plus a defective photo the deployed checks evaluate (known part; the one the damage check is most sure about). A demo choice, not a performance claim.
    nm = o.replace(" ", "_"); ex[f"{nm}_ok"] = pick(o, False)
    cand = [i for i in ite if img_df.obj[i] == o and img_df.is_defect[i] and bool(ev_final.known(emb[[i]], [o])[0])]
    flagged = [i for i in cand if float(ev_final.damage_signal(emb[[i]], [o])[0]) >= 0.5]
    ex[f"{nm}_damaged"] = max(flagged or cand, key=lambda i: float(ti.anomaly_scores(emb[[i]], [o], refs)[0]))
for n, i in ex.items(): img_df.img[i].save(f"{ART}/examples/{n}.png")
final_df[["review", "stars"]].sample(40, random_state=SEED).to_csv(f"{ART}/sample_reviews.csv", index=False)

metrics = {"deployed_sentiment": deployed, "sentiment_test": trows}
try:
    P_, R_, F_ = prf(yp, dets["tuned detector (any issue)"]); metrics["problem_detection_set3"] = {"precision": float(P_), "recall": float(R_), "F2": float(F_), "n_real": int(yp.sum()), "n_reviews": int(len(yp))}
except Exception as e: print("metrics: detector", e)
try: metrics["routing_set3"] = {"real_problems": int(len(idx)), "flagged_with_type": int(len(flagged_real)), "correct_type": int(len(hit))}
except Exception as e: print("metrics: routing", e)
try: metrics["ranking_auc_set3"] = {n: float(roc_auc_score(y, s)) for n, s in cmp_scores.items()}
except Exception as e: print("metrics: ranking", e)
try:
    wr = pd.DataFrame(res).T
    metrics["image"] = {"damage_auc": float(dm["pooled AUC"].max()), "wrong_item_auc": float(wr["AUC"].max()), "damage_flag_precision": float(DMG_P), "damage_flag_recall": float(DMG_R), "damage_check_off_for": [o for o in PRODUCTS if ANOM_THR[o] is None]}
except Exception as e: print("metrics: image", e)
json.dump(metrics, open(f"{ART}/metrics.json", "w"), indent=2, default=float)
print(json.dumps({k: v for k, v in metrics.items() if k != "sentiment_test"}, indent=2, default=float))
shutil.make_archive("triage_artifacts", "zip", ART)
print("zip size (MB):", round(os.path.getsize("triage_artifacts.zip") / 1e6))
try: files.download("triage_artifacts.zip")
except Exception as e: print("download from the Files panel (left sidebar):", e)


## 11. Limitations, ethics and honesty notes
**Data**
- Star ratings are a *proxy* label for sentiment, and the training data is **balanced by star** (real automotive reviews are about 70% five-star), so problems are more common here than in a real shop. Absolute precision would be lower in production; the *ranking* result is the transferable one.
- Issue labels (260 reviews) were produced by an AI annotator (Claude) following one written rule: a review is a *problem order* only if it describes a specific operational problem (defect, wrong item, not as described, fit / compatibility, delivery or packaging, customer service); preference or taste alone is not. The student's check of a 25-review sample is in `verify_25_labels.csv`; **agreement on problem / no problem: ____ % (fill in after checking)**. Several issue types have few examples in Set 3 (see `n_pos`), so their numbers are indicative only. Ambiguous reviews were marked low-confidence.
- Set 3 is small (100 reviews); every comparison reports a confidence interval and some gaps are not statistically distinguishable.

**Models**
- Zero-shot detection is approximate; keyword rules ignore context (praise for customer service can be flagged; "I ordered the wrong one myself" can be flagged as wrong item).
- Image models were validated on **industrial** MVTec images (seven object types standing in for auto-parts categories), not on customer snapshots, and the wrong-item test uses only 7 very different products. The damage check is switched off for part types where no cut-off reached 90% precision on the training half (Section 7a), and even where it is on, it finds only part of the damaged photos. Text+photo demos are staged.
- **A photo cannot establish whether a part fits a vehicle.** Fitment needs the part number and the vehicle's fitment data, so the tool never infers fitment from a photo; the photo check only looks for a possible defect or a wrong part type. In the app a photo counts as supporting a flag only when it shows the same kind of problem the review describes; otherwise it is reported as a separate finding. Photo scores are not calibrated probabilities, so no percentage is shown.
- Severity weights and cost figures are **assumptions** to be replaced with the retailer's own data; the sensitivity analysis shows the ranking is stable to +/-30% weight changes.

**Ethics and privacy**
- Customer photos can show faces, addresses or children: process on-device or blur personal content, and delete after triage.
- The tool *prioritises human review*; it must not auto-refund or penalise sellers/customers without a person. Monitor for bias (e.g. non-native English reviews are harder for models) and for drift when products or language change.

**Pilot plan (what to measure before claiming any business value):** actionable-flag precision (share of flags a reviewer agrees need action), missed-issue rate on a sample of unflagged reviews, review time per order, and avoidable returns. No ROI is claimed here: the dollar figures in this notebook use assumed refund shares and are for illustration only, and the app does not show them.

**Deployment status:** a prototype demonstrated from a Colab session with the models stored on Hugging Face; it is not a continuously available service.

**Next steps:** fine-tune on the retailer's own labelled reviews and return photos; replace staged pairs with real paired data; calibrate thresholds from actual cost data; add multilingual support; deploy behind an API with monitoring.

## How each resource was used and adapted
| Resource | Used for | How I adapted it |
|---|---|---|
| Course: TF-IDF + Logistic Regression (Text Analytics deck) | baseline sentiment model | class-balanced, word bigrams, judged on macro-F1 with bootstrap confidence intervals, probabilities calibrated |
| Course: BERT embeddings and transformers (Text deck) | frozen-embedding baseline, then a **fine-tuned DistilBERT** | one epoch of end-to-end fine-tuning, isotonic calibration, word-deletion explanations |
| Course: CNN in Keras (Image deck) | CNN baseline on defect photos | same architecture recipe, compared with transfer learning |
| Course: ViT/embeddings + classifier (Image deck) | CLIP image embeddings + Logistic Regression | CLIP instead of ViT; reference-photo comparison for wrong-item detection |
| Course: Gradio / Render deployment (Text deck) | the business web app | redesigned around a manager's decisions; hosted in the cloud |
| Hugging Face zero-shot DeBERTa model | find issue types without labelled data | thresholds chosen by leave-one-out F2; combined with keyword rules |
| Amazon Reviews 2023, Automotive (McAuley Lab) | all review text: sentiment training, issue labels, demo | streamed and split by position so training, labelling and demo data never overlap |
| MVTec AD defect images | photo data (7 object types mapped to auto parts) | industrial objects: validates the method, limits stated |
| Complaint-detection research (Singh et al., 2022) | motivation for combining text and images | informed the combined flag |

## References
- Hou, Y., Li, J., He, Z., Yan, A., Chen, X., McAuley, J. (2024). *Bridging Language and Items for Retrieval and Recommendation* (Amazon Reviews 2023). arXiv:2403.03952. (data: Hugging Face `McAuley-Lab/Amazon-Reviews-2023`, category Automotive)
- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12.
- Reimers, N., Gurevych, I. (2019). *Sentence-BERT.* EMNLP. (`all-MiniLM-L6-v2`, `clip-ViT-B-32` via sentence-transformers)
- Sanh, V. et al. (2019). *DistilBERT, a distilled version of BERT.* arXiv:1910.01108.
- He, P. et al. (2021). *DeBERTa: Decoding-enhanced BERT with Disentangled Attention.* ICLR; zero-shot model `MoritzLaurer/deberta-v3-base-zeroshot-v2.0` (Hugging Face model card).
- Radford, A. et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision (CLIP).* ICML.
- Bergmann, P. et al. (2019). *MVTec AD: A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection.* CVPR. (Hugging Face mirror `TheoM55/mvtec_all_objects_split`)
- Niculescu-Mizil, A., Caruana, R. (2005). *Predicting Good Probabilities with Supervised Learning.* ICML. (isotonic calibration)
- Singh, A. et al. (2022). *Sentiment and Emotion-Aware Multi-Modal Complaint Identification.* AAAI. (motivation: images help complaint detection)
- Course materials, BC3415 AI in Accounting and Finance (Teoh Teik Toe): *Text Analytics & Transformer*; *Image Processing & CNN & Transformer* (TF-IDF, BERT embeddings, ViT, Gradio).
- Abid, A. et al. *Gradio: Hassle-free sharing and testing of ML models in the wild.*
